# Kvartalsvis pro forma-resultaträkning med PROC COMPUTAB

## Sammanfattning

Denna notebook bygger en regionalbanks kvartalsvisa pro forma-resultaträkning direkt från månatlig huvudboksdata med PROC COMPUTAB, tabellrapportproceduren i SAS/ETS. Den dirigerar varje månads ränteintäkter, räntekostnader, avgiftsintäkter och driftskostnader till rätt kalenderkvartalskolumn, och använder sedan programmerbara radblock för att beräkna räntenetto, resultat före skatt och nettoresultat, samt ett kolumnblock för att rulla ihop de fyra kvartalen till en helårssumma.

## Datakällor

Det enda syntetiska datasetet `bank_ledger` simulerar ett räkenskapsår av månatliga poster i finansiella rapporter för en medelstor lokalbank. Tolv månatliga observationer genereras direkt i koden med `call streaminit`/`rand` så att notebooken är helt fristående.

| Variabel | Typ | Beskrivning |
|----------|------|-------------|
| `stmtdate` | Num (DATE9.) | Månadsslutets rapportdatum (jan–dec) |
| `loanint`  | Num | Ränta och avgifter intjänade på låneportföljen (tusental USD) |
| `secint`   | Num | Ränta intjänad på placeringsvärdepappersboken (tusental USD) |
| `depint`   | Num | Ränta betald på inlåning (tusental USD) |
| `borrint`  | Num | Ränta betald på upplåning / FHLB-förskott (tusental USD) |
| `feeinc`   | Num | Icke-ränteintäkter (avgifts- och serviceintäkter) (tusental USD) |
| `salaries` | Num | Löner och personalförmånskostnader (tusental USD) |
| `occupancy`| Num | Lokal- och utrustningskostnad (tusental USD) |
| `othropex` | Num | Övriga icke-ränterelaterade driftskostnader (tusental USD) |
| `provision`| Num | Avsättning för kreditförluster (tusental USD) |
| `taxrate`  | Num | Effektiv skattesats tillämpad på resultat före skatt |

# Kvartalsvis pro forma-resultaträkning med PROC COMPUTAB

Bankers finansteam förvandlar rutinmässigt en månatlig huvudbok till en **kvartalsvis resultaträkning** som visar räntenetto och nettoresultat på sista raden. `PROC COMPUTAB` (SAS/ETS) är byggd för just detta: den lägger ut en programmerbar tabell vars **kolumner** är rapportperioderna och vars **rader** är rapportens radposter, och den låter dig beräkna delsummor med vanlig DATA-stegslogik i rad- och kolumnblock.

I denna notebook:

1. Genererar vi ett räkenskapsår av syntetisk månatlig huvudboksdata för en lokalbank.
2. Dirigerar vi varje månad till dess kalenderkvartal och bygger en kvartalsvis resultaträkning.
3. Beräknar vi räntenetto, resultat före skatt och nettoresultat i ett **radblock**, och rullar ihop kvartalen till en helårssumma i ett **kolumnblock**.
4. Återanvänder vi `out=`-tabellen så att den beräknade rapporten kan mata efterföljande rapportering.

## Steg 1 — Generera syntetisk månatlig huvudboksdata

Vi simulerar tolv observationer vid månadsslut. Ränteintäkter från lån och värdepapper trendar svagt uppåt över året, kostnaderna för inlåning och upplåning skalar med ränteläget, och avgiftsintäkter, driftskostnader och kreditförlustavsättningen bär realistiskt brus från månad till månad. `call streaminit` fixerar fröet så att data är reproducerbara.

In [1]:
data bank_ledger;
   CALL streaminit(20240115);
   format stmtdate date9.;
   GÖR month = 1 TILL 12;
      /* Month-end statement date for fiscal year 2024 */
      stmtdate = mdy(month, 28, 2024);

      /* Mild upward drift across the year + monthly noise */
      drift = 1 + 0.012 * (month - 1);

      /* Interest income (USD thousands) */
      loanint = round(1850 * drift + rand('normal', 0, 45), 0.01);
      secint  = round( 420 * drift + rand('normal', 0, 15), 0.01);

      /* Interest expense (USD thousands) */
      depint  = round( 540 * drift + rand('normal', 0, 22), 0.01);
      borrint = round( 130 * drift + rand('normal', 0, 10), 0.01);

      /* Non-interest income and expense (USD thousands) */
      feeinc    = round(310 + rand('normal', 0, 18), 0.01);
      salaries  = round(720 + rand('normal', 0, 25), 0.01);
      occupancy = round(165 + rand('normal', 0, 8),  0.01);
      othropex  = round(240 + rand('normal', 0, 20), 0.01);

      /* Provision for credit losses, occasionally elevated */
      provision = round(95 + rand('exponential') * 40, 0.01);

      /* Effective tax rate */
      taxrate = 0.21;

      UTDATA;
   SLUT;
   BEHÅLL stmtdate loanint secint depint borrint
        feeinc salaries occupancy othropex provision taxrate;
KÖR;

PROCEDUR SKRIV data=bank_ledger noobs ETIKETT;
   TITEL 'Synthetic Monthly Ledger — Community Bank FY2024';
   format loanint secint depint borrint feeinc
          salaries occupancy othropex provision comma8.2;
KÖR;

                                    Synthetic Monthly Ledger — Community Bank FY2024                                    

 STMTDATE   LOANINT  SECINT  DEPINT  BORRINT  FEEINC  SALARIES  OCCUPANCY  OTHROPEX  PROVISION  TAXRATE
28JAN2024  1,772.95  423.79  531.47   128.99  339.33    699.38     171.95    202.43     130.93     0.21
28FEB2024  1,824.38  421.13  564.85   125.53  294.09    767.29     162.79    307.61     123.25     0.21
28MAR2024  1,931.98  442.06  536.64   131.71  295.72    724.03     153.26    254.21     115.76     0.21
28APR2024  1,934.99  439.29  535.94   140.01  294.56    729.47     174.19    237.43     198.05     0.21
28MAY2024  1,949.31  447.35  591.44   124.42  299.78    739.13     165.08    223.32     105.57     0.21
28JUN2024  1,934.36  448.28  551.70   147.64  335.81    718.90     171.91    236.94     130.13     0.21
28JUL2024  1,936.57  439.22  565.70   133.82  293.21    743.87     174.16    247.19     199.20     0.21
28AUG2024  1,979.73  457.42  558.54   144.45  

## Steg 2 — Bygg den kvartalsvisa resultaträkningen

Hjärtat i rapporten är `PROC COMPUTAB`-steget nedan.

- **`columns qtr1 qtr2 qtr3 qtr4 fy;`** definierar fyra kvartalskolumner plus en helårskolumn.
- Det omärkta **indatablocket** sätter den automatiska variabeln **`_col_`** med `qtr(stmtdate)`, vilket dirigerar varje månatlig observation till rätt kvartalskolumn. Eftersom indata transponeras som standard hamnar varje huvudboksvariabel på den rad som delar dess namn.
- **Radblocket** `inc_stmt:` körs en gång per kolumn och beräknar de härledda raderna — räntenetto, totala icke-ränterelaterade kostnader, resultat före skatt och nettoresultat — precis som en redovisare skulle göra.
- **Kolumnblocket** `total:` körs en gång per rad och summerar de fyra kvartalen till `fy`-kolumnen (helår).

Satserna `rows ... / ...` lägger till titlar, indrag och linjeregler (`ol` överlinje, `ul` underlinje, `dul` dubbel underlinje) så att utdata läses som en riktig resultaträkning.

In [2]:
TITEL 'Pro Forma Income Statement — Community Bancorp, Inc.';
title2 'Fiscal Year 2024  (Amounts in USD Thousands)';

PROCEDUR computab data=bank_ledger cspace=2 cwidth=11 out=qtr_income;

   /* Four quarters plus a rolled-up full-year column */
   columns qtr1 qtr2 qtr3 qtr4 fy / format=comma11.1;
   columns qtr1 / 'Q1';
   columns qtr2 / 'Q2';
   columns qtr3 / 'Q3';
   columns qtr4 / 'Q4';
   columns fy   / 'Full' 'Year' +3;

   /* Income statement rows, top to bottom */
   rows loanint  / 'Interest & Fees on Loans';
   rows secint   / 'Interest on Securities'        ul;
   rows intinc   / 'Total Interest Income';
   rows depint   / 'Interest on Deposits';
   rows borrint  / 'Interest on Borrowings'        ul;
   rows intexp   / 'Total Interest Expense';
   rows nii      / 'Net Interest Income'           ol skip;
   rows provision/ 'Provision for Credit Losses'   ul;
   rows niiap    / 'Net Int. Income after Prov.'   skip;
   rows feeinc   / 'Non-Interest Income'           ul;
   rows salaries / 'Salaries & Benefits';
   rows occupancy/ 'Occupancy & Equipment';
   rows othropex / 'Other Operating Expense'       ul;
   rows nonintexp/ 'Total Non-Interest Expense'    skip;
   rows pretax   / 'Pre-Tax Income'                ol;
   rows taxexp   / 'Income Tax Expense'            ul;
   rows netinc   / 'Net Income'                    dul;

   /* Input block: route each month to its calendar quarter */
   _col_ = qtr(stmtdate);

   /* Row block: compute statement subtotals across every column */
   inc_stmt:
      intinc    = loanint + secint;
      intexp    = depint + borrint;
      nii       = intinc - intexp;
      niiap     = nii - provision;
      nonintexp = salaries + occupancy + othropex;
      pretax    = niiap + feeinc - nonintexp;
      taxexp    = pretax * 0.21;
      netinc    = pretax - taxexp;

   /* Column block: roll the four quarters into the full-year column */
   TOTAL:
      fy = qtr1 + qtr2 + qtr3 + qtr4;
KÖR;

                                  Pro Forma Income Statement — Community Bancorp, Inc.                                  
                                      Fiscal Year 2024  (Amounts in USD Thousands)                                      


                             The COMPUTAB Procedure                             

                             Q1           Q2           Q3           Q4         Full  
                                                                               Year  
                           qtr1         qtr2         qtr3         qtr4           fy  
                    -----------  -----------  -----------  -----------  -----------  
  Interest & Fees on Loans
  loanint               5529.31      5818.66      5963.46      6276.96     23588.39  
  Interest on Securities
  secint                1286.98      1334.92      1342.03      1452.88      5416.81  
                    -----------  -----------  -----------  -----------  -----------  
  Total Interest Inc

## Steg 3 — Återanvänd COMPUTAB-utdatadatasetet

`PROC COMPUTAB`-steget ovan skrev sin beräknade tabell till `qtr_income` via `out=`. Varje rad i det datasetet är en radpost i rapporten och varje kolumnvariabel (`qtr1`–`qtr4`, `fy`) håller det beräknade värdet, så det kan mata efterföljande rapportering. Nedan skriver vi ut den ihoprullade helårskolumnen för att bekräfta att siffrorna stämmer.

In [3]:
TITEL 'COMPUTAB Output Dataset — Full-Year Column';

PROCEDUR SKRIV data=qtr_income ETIKETT noobs;
   VARIABEL _row_ fy;
   format fy comma12.1;
   ETIKETT _row_ = 'Line Item' fy = 'Full Year';
KÖR;

TITEL;

                                       COMPUTAB Output Dataset — Full-Year Column                                       
                                      Fiscal Year 2024  (Amounts in USD Thousands)                                      


Line Item  Full Year
---------  ---------
loanint     23,588.4
secint       5,416.8
intinc      29,005.2
depint       6,831.2
borrint      1,650.7
intexp       8,482.0
nii         20,523.2
provision    1,568.9
niiap       18,954.3
feeinc       3,703.2
salaries     8,763.1
occupancy    1,985.6
othropex     2,944.8
nonintexp   13,693.5
pretax       8,964.1
taxexp       1,882.5
netinc       7,081.6

NOTE: Option TITLE changed to COMPUTAB Output Dataset — Full-Year Column.
NOTE: PROC PRINT data=qtr_income

NOTE: PROC PRINT completed: 17 observations printed, 2 variables


## Tolka resultaten

Pro forma-rapporten läses uppifrån och ner som en banks tillsynsmässiga call report: totala ränteintäkter minus totala räntekostnader ger **räntenetto** — här omkring \$20,5 miljoner för året — bankens främsta intjäningsdrivkraft. Att subtrahera **avsättningen för kreditförluster**, lägga till **avgiftsintäkter** och räkna av **icke-ränterelaterade kostnader** ger ett resultat före skatt på ungefär \$9,0 miljoner, och att tillämpa den effektiva skattesatsen på 21 % ger ett **nettoresultat** nära \$7,1 miljoner. Kolumnblocket `total:` adderar de fyra kvartalen till helårskolumnen, så årssummorna stämmer med summan av kvartalen per konstruktion — bekräftat i `out=`-datasetet, där helårets `netinc` på 7 081,6 är lika med summan av de fyra kvartalssiffrorna.

Eftersom allt beräknas inne i `PROC COMPUTAB`:s programmerbara rad- och kolumnblock kräver ett byte till en verklig månatlig huvudbok ingen ändring av rapportlogiken — endast indatadatasetet ändras. `out=`-datasetet gör sedan den beräknade rapporten tillgänglig för instrumentpaneler, trendanalys eller automatisering av styrelseunderlag.